In [ ]:
!pip install torch torchvision pennylane

In [ ]:
import torch
import torch.nn as nn
import pennylane as qml
import numpy as np

# ==========================================
# 1. Constants & Configuration
# ==========================================
N_QUBITS = 12
N_ELEMENTS = 9
N_PATCHES = 64  # (32/4)^2
POOL_SIZE = 4

# Qubit Mapping
LOC_QUBITS = list(range(6))      # Wires 0-5
ELEM_QUBITS = list(range(6, 9))  # Wires 6-8
KERNEL_QUBIT = [9]               # Wire 9
READOUT_QUBITS = [10, 11]        # Wires 10-11

# Define device
# 'lightning.qubit' is recommended for performance
dev = qml.device("lightning.qubit", wires=N_QUBITS)

# ==========================================
# 2. Gate Definitions (The Corrected Logic)
# ==========================================

def core_u3(theta, phi, lam, target_wire):
    """Standard U3 rotation sequence."""
    qml.RZ(lam, wires=target_wire)
    qml.RX(np.pi/2, wires=target_wire)
    qml.RZ(theta, wires=target_wire)
    qml.RX(-np.pi/2, wires=target_wire)
    qml.RZ(phi, wires=target_wire)

def apply_encoding_u3(theta, phi, lam, target_wire, loc_qubits, ctrl_state):
    """
    Reverse order control for Encoding to match Big-Endian spatial mapping.
    """
    reversed_loc = loc_qubits[::-1]
    qml.ctrl(core_u3, control=reversed_loc, control_values=ctrl_state)(
        theta, phi, lam, target_wire
    )

def apply_convolution_cu3(theta, phi, lam, target_wire, control_wires, ctrl_state):
    """
    Forward order control for Convolution to match specific kernel flags.
    """
    qml.ctrl(core_u3, control=control_wires, control_values=ctrl_state)(
        theta, phi, lam, target_wire
    )

def get_binary_control_values(idx, num_bits):
    """Helper: int -> binary list"""
    return [int(b) for b in format(idx, f'0{num_bits}b')]

# ==========================================
# 3. The Quantum Circuit
# ==========================================

@qml.qnode(dev, interface="torch")
def seqnn_circuit(inputs, weights):
    
    # --- A. Initialization ---
    for q in LOC_QUBITS:
        qml.Hadamard(wires=q)
    for q in KERNEL_QUBIT:
        qml.Hadamard(wires=q)

    # --- B. Encoding (Superpixel) ---
    for i in range(N_PATCHES):
        ctrl_values = get_binary_control_values(i, 6)
        base_idx = i * N_ELEMENTS
        
        # Apply encoding to the 3 Element Qubits
        for j, target_w in enumerate(ELEM_QUBITS):
            # Each element needs 3 input values (theta, phi, lam)
            p_idx = base_idx + (j * 3)
            apply_encoding_u3(inputs[p_idx], inputs[p_idx+1], inputs[p_idx+2], 
                              target_w, LOC_QUBITS, ctrl_values)

        # Entanglement ring
        qml.CZ(wires=[ELEM_QUBITS[0], ELEM_QUBITS[1]])
        qml.CZ(wires=[ELEM_QUBITS[1], ELEM_QUBITS[2]])
        qml.CZ(wires=[ELEM_QUBITS[2], ELEM_QUBITS[0]])

    # --- C. Quantum Convolution (QDCNN) ---
    w_idx = 0
    
    # Layer Definitions based on notebook `qdcnn` calls
    # Format: (Target_Wire, Readout_Source_0, Readout_Source_1, Wire_Shift_Index)
    # Wire_Shift_Index: 0 means use LOC[0] & LOC[3]. 1 means use LOC[1] & LOC[4].
    layer_configs = [
        (ELEM_QUBITS[0], READOUT_QUBITS[0], 0),   # Layer 1 (shift 0)
        (READOUT_QUBITS[0], READOUT_QUBITS[1], 1),# Layer 2 (shift 1)
        (ELEM_QUBITS[1], READOUT_QUBITS[0], 0),   # Layer 3 (shift 0)
        (READOUT_QUBITS[0], READOUT_QUBITS[1], 1),# Layer 4 (shift 1)
        (ELEM_QUBITS[2], READOUT_QUBITS[0], 0),   # Layer 5 (shift 0)
        (READOUT_QUBITS[0], READOUT_QUBITS[1], 1) # Layer 6 (shift 1)
    ]

    for (input_q, output_q, shift) in layer_configs:
        
        # Determine specific control wires for this layer (The Wire Shift Fix)
        # In notebook: yloc=qubits[0+shift], xloc=qubits[3+shift]
        yloc_wire = LOC_QUBITS[0 + shift]
        xloc_wire = LOC_QUBITS[3 + shift]
        kern_wire = KERNEL_QUBIT[0]
        
        # The 4 control wires in Forward Order: [y, x, target, kernel]
        specific_controls = [yloc_wire, xloc_wire, input_q, kern_wire]
        
        # Iterate over kernel states (0 and 1)
        for k_val in [0, 1]:
            # Iterate over location states (00, 10, 01, 11)
            loc_states = [[0,0], [1,0], [0,1], [1,1]]
            
            for ls in loc_states:
                # Get weights
                theta, phi, lam = weights[w_idx], weights[w_idx+1], weights[w_idx+2]
                w_idx += 3
                
                # Construct control values: loc_state + [1] + [k_val]
                # [1] is the control on the target/input qubit
                specific_values = ls + [1] + [k_val]
                
                apply_convolution_cu3(theta, phi, lam, output_q, 
                                      specific_controls, specific_values)

    # --- D. Measurement ---
    # We return Z expectations on all 12 qubits to form the feature vector
    return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

# ==========================================
# 4. PyTorch Modules
# ==========================================

class SuperpixelLayer(nn.Module):
    def __init__(self, in_channels, n_elements=9, pool_size=4):
        super().__init__()
        # Matches `tf.image.extract_patches`
        self.unfold = nn.Unfold(kernel_size=pool_size, stride=pool_size)
        
        # Matches the `w` and `b` weights in `Superpixel` class
        # Input dim: channels * 4 * 4
        patch_dim = in_channels * pool_size * pool_size
        self.projection = nn.Linear(patch_dim, n_elements)
        self.relu = nn.ReLU()

    def forward(self, x):
        # x: (Batch, C, 32, 32)
        b_size = x.shape[0]
        
        # Extract patches -> (Batch, Patch_Dim, Num_Patches)
        patches = self.unfold(x)
        
        # Transpose for Linear -> (Batch, Num_Patches, Patch_Dim)
        patches = patches.transpose(1, 2)
        
        # Project -> (Batch, Num_Patches, n_elements)
        out = self.projection(patches)
        out = self.relu(out)
        
        # Flatten for quantum circuit -> (Batch, Num_Patches * n_elements)
        # (Batch, 64 * 9) = (Batch, 576)
        return out.reshape(b_size, -1)

class SEQNN(nn.Module):
    def __init__(self, num_classes, in_channels=4):
        super().__init__()
        
        # 1. Classical Preprocessing
        self.superpixel = SuperpixelLayer(in_channels, N_ELEMENTS, POOL_SIZE)
        
        # 2. Quantum Layer
        # Weights shape: 144 (6 layers * 2 kernel_states * 4 loc_states * 3 params)
        weight_shapes = {"weights": (144,)}
        self.q_layer = qml.qnn.TorchLayer(seqnn_circuit, weight_shapes)
        
        # 3. Classifier
        # Input 12 comes from measuring 12 qubits. 
        # (Original TFQ had 64 complex correlations, simplified here to 12 dense features)
        self.classifier = nn.Linear(12, num_classes)

    def forward(self, x):
        # x input is (Batch, 32, 32, C). PyTorch expects (Batch, C, 32, 32)
        x = x.permute(0, 3, 1, 2)
        
        x = self.superpixel(x)
        x = self.q_layer(x)
        x = self.classifier(x)
        return x

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
# Assumes seqnn_dataLoader.py is in the same folder
from seqnn_dataLoader import DataLoader as OriginalLoader

class SEQNN_Dataset(Dataset):
    def __init__(self, dataset_name, split='train'):
        loader = OriginalLoader(dataset_name)
        self.categories = loader.get_categories()
        
        if split == 'train':
            self.x, self.y = loader.train_x, loader.train_y
        elif split == 'valid':
            self.x, self.y = loader.valid_x, loader.valid_y
        elif split == 'test':
            self.x, self.y = loader.test_x, loader.test_y
        
        # Fix Labels: TFQ loader returns One-Hot. We need Indices for PyTorch.
        # Check if labels are already one-hot (2D array)
        if len(self.y.shape) > 1 and self.y.shape[1] > 1:
            self.y = np.argmax(self.y, axis=1)
            
        # Fix Data Type
        self.x = self.x.astype(np.float32)
        self.y = self.y.astype(np.longlong)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return torch.from_numpy(self.x[idx]), torch.tensor(self.y[idx])

def get_dataloaders(dataset_name, batch_size=32):
    train_ds = SEQNN_Dataset(dataset_name, 'train')
    val_ds = SEQNN_Dataset(dataset_name, 'valid')
    test_ds = SEQNN_Dataset(dataset_name, 'test')
    
    # Drop_last=True helps prevent shape errors with batch normalization if used
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_dl = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    
    return train_dl, val_dl, test_dl, len(train_ds.categories)